# Image Quality Assessment Training

Reusable Google Colab training infrastructure for this stage of the diabetic
retinopathy pipeline (see `PROJECT_CODE.md` / `IMPLEMENTATION_PLAN.md` in the
repository root for the full target architecture).

**Pipeline stage:** 1. Image Quality Assessment
**Target model (per `PROJECT_CODE.md`):** EfficientNetB0 (quality classifier)
**Classes:** Good / Usable / Reject (EyeQ `quality` column: 0 / 1 / 2)

> Dataset loading and the model architecture are implemented in
> `image_quality_dataset.py` and `image_quality_model.py` respectively (see
> repository root) and imported here in Sections 7 and 8 -- this notebook
> wires them into the shared GPU/mixed-precision setup, dependency install,
> project setup, dataset path resolution, checkpointing, resume support,
> early stopping, LR scheduling, TensorBoard, and weight-export
> infrastructure that was already scaffolded for every training stage.

**Runtime:** Runtime > Change runtime type > Hardware accelerator > GPU (T4 or better recommended).

## 1. GPU Runtime Check

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {[g.name for g in gpus]}")
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except RuntimeError as e:
            print(f"Could not set memory growth on {g.name}: {e}")
else:
    print(
        "No GPU detected. Go to Runtime > Change runtime type > "
        "Hardware accelerator > GPU, then re-run this cell."
    )

## 2. Project Setup

Clones this repository directly into the Colab VM's local disk (`/content`) --
NOT Google Drive. Datasets will also live inside this cloned copy, under
`datasets/`, uploaded directly into the Colab session.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Cloning {REPO_URL} (branch={BRANCH}) into {REPO_DIR} ...")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already present at {REPO_DIR}; pulling latest {BRANCH} ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "notebooks")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project setup complete. Repository root:", REPO_DIR)

## 3. Install Dependencies

In [ ]:
import colab_utils

colab_utils.install_requirements(REPO_DIR)

## 4. Mixed Precision

In [ ]:
mixed_precision_policy = colab_utils.setup_mixed_precision()

## 5. Dataset Path Configuration

Datasets are expected to be uploaded directly into the Colab VM's copy of the
repository, under `datasets/EyeQ` -- not mounted from Google Drive.

EyeQ is the quality-labeled dataset called for in `PROJECT_CODE.md`.

In [ ]:
MODULE_KEY = "image_quality_assessment"
DATASET_SUBFOLDER = "EyeQ"  # matches datasets/EyeQ used locally in this repo

DATASET_DIR = colab_utils.resolve_dataset_dir(REPO_DIR, DATASET_SUBFOLDER)

## 6. Training Configuration

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 1e-4
FREEZE_LAYERS = 100  # number of leading EfficientNet-B0 backbone layers to freeze
RESUME_TRAINING = False  # set True to continue from the last checkpoint of a previous run

RUN_DIR = f"/content/training_runs/{MODULE_KEY}"
os.makedirs(RUN_DIR, exist_ok=True)

print("Training configuration:")
print(f"  IMAGE_SIZE = {IMAGE_SIZE}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  EPOCHS = {EPOCHS}")
print(f"  LEARNING_RATE = {LEARNING_RATE}")
print(f"  FREEZE_LAYERS = {FREEZE_LAYERS}")
print(f"  RUN_DIR = {RUN_DIR}")

## 7. Dataset Loading

In [ ]:
from image_quality_dataset import load_eyeq_datasets

CLASS_WEIGHTS = None  # populated by load_dataset() below, used by the training loop in Section 11

def load_dataset(dataset_dir, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE):
    """
    Loads the EyeQ Image Quality Assessment dataset directly from
    datasets/EyeQ/raw/train (stratified train/val split, RGB images, no
    CLAHE/Ben Graham/other preprocessing -- see image_quality_dataset.py).

    Returns (train_ds, val_ds); also stores the training-split class_weight
    dict on the global CLASS_WEIGHTS, since EyeQ's quality labels are
    imbalanced (Good is the majority class).
    """
    global CLASS_WEIGHTS
    raw_dir = os.path.join(dataset_dir, "raw")
    train_ds, val_ds, CLASS_WEIGHTS = load_eyeq_datasets(
        raw_dir=raw_dir, image_size=image_size, batch_size=batch_size,
    )
    print(f"Class weights (train split): {CLASS_WEIGHTS}")
    return train_ds, val_ds

## 8. Model Definition

In [ ]:
from image_quality_model import build_iqa_model

def build_model(input_shape, learning_rate=LEARNING_RATE, freeze_layers=FREEZE_LAYERS):
    """
    Builds the EfficientNet-B0 Image Quality Assessment classifier --
    ImageNet-pretrained backbone (first `freeze_layers` layers frozen),
    GAP + dense head, 3-way Good/Usable/Reject softmax output. See
    image_quality_model.py for the architecture.
    """
    return build_iqa_model(input_shape=input_shape, learning_rate=learning_rate, freeze_layers=freeze_layers)

## 9. Callbacks, Checkpointing & Resume Training Check

In [ ]:
callbacks, checkpoint_paths = colab_utils.build_training_callbacks(RUN_DIR, save_weights_only=False)

initial_epoch = 0
if RESUME_TRAINING:
    initial_epoch = colab_utils.get_resume_epoch(checkpoint_paths["epoch_state"])
    if initial_epoch > 0 and os.path.exists(checkpoint_paths["last_weights"]):
        print(f"Will resume from epoch {initial_epoch} using {checkpoint_paths['last_weights']}")
    else:
        print("RESUME_TRAINING=True but no prior checkpoint was found; starting from scratch.")
        initial_epoch = 0
else:
    print("RESUME_TRAINING=False; starting from scratch.")

## 10. TensorBoard

In [ ]:
logs_dir = checkpoint_paths["logs_dir"]
%load_ext tensorboard
%tensorboard --logdir $logs_dir

## 11. Training Loop

In [ ]:
train_ds, val_ds = load_dataset(DATASET_DIR, IMAGE_SIZE, BATCH_SIZE)
model = build_model(input_shape=(*IMAGE_SIZE, 3), learning_rate=LEARNING_RATE)

if initial_epoch > 0:
    model.load_weights(checkpoint_paths["last_weights"])

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    callbacks=callbacks,
    class_weight=CLASS_WEIGHTS,
)

## 12. Validation

In [ ]:
val_results = colab_utils.evaluate_and_report(model, val_ds)
colab_utils.plot_history(history, output_path=os.path.join(RUN_DIR, "training_history.png"))

## 13. Save Best Model

The best checkpoint (by `val_loss`) was already saved automatically during
training via the `ModelCheckpoint` callback in Section 9.

In [ ]:
best_weights_path = checkpoint_paths["best_weights"]
assert os.path.exists(best_weights_path), "No best checkpoint found -- did training run?"
print(f"Best model weights: {best_weights_path}")
print(f"Size: {os.path.getsize(best_weights_path) / 1e6:.2f} MB")

## 14. Export Weights to Repository

In [ ]:
exported_path = colab_utils.export_trained_model(
    best_weights_path=best_weights_path,
    repo_dir=REPO_DIR,
    module_key=MODULE_KEY,
    filename="best_model.keras",
)
print(f"Exported to {exported_path}")

## 14b. Full Evaluation on Held-Out Test Split

Runs the exported model over `datasets/EyeQ/raw/test/` (never used for
training or validation above) and reports accuracy, precision, recall, F1,
confusion matrix, classification report, and ROC/AUC via the reusable
`evaluation/` framework -- see `evaluate_image_quality.py` in the repository
root.

In [ ]:
from evaluate_image_quality import evaluate

test_report = evaluate(
    raw_dir=os.path.join(DATASET_DIR, "raw"),
    model_path=exported_path,
    batch_size=BATCH_SIZE,
    output_dir=os.path.join(RUN_DIR, "evaluation"),
)

## 15. (Optional) Commit & Push Weights

**Disabled by default.** Committing and pushing trained weights from a Colab
session is a hard-to-reverse, shared-repository action -- review the exported
file yourself before enabling this. Set `DO_COMMIT_AND_PUSH = True` and, if you
also want it pushed to GitHub immediately, `DO_PUSH = True`, then re-run this
cell. Otherwise, download the exported file from Section 14 and commit it
yourself from your local machine.

In [ ]:
DO_COMMIT_AND_PUSH = False
DO_PUSH = False

if DO_COMMIT_AND_PUSH:
    colab_utils.git_commit_and_push(
        REPO_DIR,
        message=f"Add trained {MODULE_KEY} weights",
        paths=[os.path.relpath(exported_path, REPO_DIR)],
        push=DO_PUSH,
    )
else:
    print("Skipped -- set DO_COMMIT_AND_PUSH = True to enable (see markdown above).")